# dbt Docs Agent — Full Pipeline

The complete system, top to bottom, in one kernel: **ingest → chunk → index →
agent → evaluate**. This is the *product* notebook. The Day 1–5 notebooks are the
*journey* (with all the narration, dead ends, and findings) — keep those as the
learning record; this is the clean runnable pipeline.

**What's new here vs. the day notebooks:** chunking is **header-first** (split on
`##`/`###`), which fixes the Day 5 finding — sliding windows had smeared the
Wizard "Build a new model" answer across 11 near-duplicate fragments. Header-first
isolates it into one clean chunk.

Run every cell in order.

In [1]:
# uv add requests python-frontmatter minsearch sentence-transformers anthropic python-dotenv pandas numpy

In [2]:
import io, os, re, json, time, random, zipfile, statistics
import numpy as np
import pandas as pd
import requests, frontmatter
from dotenv import load_dotenv
import anthropic
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer

load_dotenv()
client = anthropic.Anthropic()

OWNER, REPO, BRANCH = "dbt-labs", "docs.getdbt.com", "current"
ZIP_CACHE = "repo.zip"
CHUNKS_FILE = "chunks_headerfirst.jsonl"           # new file; keeps old one intact
MODEL = "claude-haiku-4-5-20251001"
EMB_MODEL_NAME = "multi-qa-distilbert-cos-v1"
EMB_CACHE = f"embeddings_headerfirst_{EMB_MODEL_NAME.replace('/','_')}.npy"
QUESTIONS_FILE = "eval_questions.json"
RESULTS_FILE = "eval_results_headerfirst.json"

## 1. Ingest (Day 1)

In [3]:
def download_repo():
    if not os.path.exists(ZIP_CACHE):
        url = f"https://codeload.github.com/{OWNER}/{REPO}/zip/refs/heads/{BRANCH}"
        r = requests.get(url, timeout=300); r.raise_for_status()
        open(ZIP_CACHE, "wb").write(r.content)
    return zipfile.ZipFile(ZIP_CACHE)

def strip_root(p): return p.split("/", 1)[1] if "/" in p else p

zf = download_repo()
documents = []
for info in zf.infolist():
    if not info.filename.lower().endswith((".md", ".mdx")):
        continue
    post = frontmatter.loads(zf.open(info).read().decode("utf-8", errors="ignore"))
    d = post.to_dict()
    d["filename"] = strip_root(info.filename)
    documents.append(d)
print(f"documents: {len(documents)}")

documents: 1547


## 2. Chunk — HEADER-FIRST (the Day 5 fix)

Split on `##`, then `###` for oversized sections. Each chunk is prefixed with
`[doc title]` for context but keeps its own distinct section header + body — so
chunks from the same file don't collapse into look-alikes (the bug that smeared
the Wizard answer across 11 fragments).

In [4]:
IMPORT_RE=re.compile(r"^\s*(?:import|export)\s+[^\n]*\n",re.M)
COMPONENT_RE=re.compile(r"""</?[A-Z][A-Za-z0-9]*(?:\s(?:"[^"]*"|'[^']*'|[^<>"'])*?)?/?>""")
SNIPPET_RE=re.compile(r"<Snippet\s+[^>]*path=[\"']([^\"']+)[\"'][^>]*/?>")
CONST_RE=re.compile(r'<Constant\s+name=["\']([^"\']+)["\']\s*/>')
HTML_COMMENT=re.compile(r"<!--.*?-->",re.S)
ADM_OPEN=re.compile(r"^:::+\s*\w*\s*(.*)$",re.M); ADM_CLOSE=re.compile(r"^:::+\s*$",re.M)
FENCE=re.compile(r"(```.*?```|~~~.*?~~~)",re.S); BLANKS=re.compile(r"\n{3,}")

def clean_prose(t):
    t=HTML_COMMENT.sub("",t); t=CONST_RE.sub(r"\1",t)      # <Constant name="wizard"/> -> wizard
    t=IMPORT_RE.sub("",t); t=COMPONENT_RE.sub("",t)
    t=ADM_CLOSE.sub("",t); t=ADM_OPEN.sub(r"\1",t); return t

def clean_mdx(t):
    parts=FENCE.split(t)   # never clean inside code fences
    return BLANKS.sub("\n\n","".join(p if i%2 else clean_prose(p) for i,p in enumerate(parts))).strip()

def split_by_level(text, level):
    pat=re.compile(r"^(#{"+str(level)+r"}\s+.+)$",re.M); parts=pat.split(text)
    secs=[]
    if parts[0].strip(): secs.append(parts[0].strip())     # keep preamble (the 19% fix)
    for i in range(1,len(parts),2):
        h=parts[i].strip(); c=parts[i+1].strip() if i+1<len(parts) else ""
        secs.append(h+"\n\n"+c)
    return secs

def chunk_document(doc, max_chars=2000):
    meta=doc.copy(); content=clean_mdx(meta.pop("content"))
    title=str(meta.get("title") or meta["filename"].rsplit("/",1)[-1].rsplit(".",1)[0])
    chunks=[]
    for sec in split_by_level(content, 2):
        pieces=[sec] if len(sec)<=max_chars else split_by_level(sec, 3)
        for p in pieces:
            if not p.strip(): continue
            rec=dict(meta)
            rec["chunk"]=f"[{title}] {p}".strip()   # context prefix + distinct body
            chunks.append(rec)
    return chunks

dbt_chunks=[]
for doc in documents:
    dbt_chunks.extend(chunk_document(doc))

with open(CHUNKS_FILE,"w",encoding="utf-8") as out:
    for c in dbt_chunks:
        out.write(json.dumps(c, default=str, ensure_ascii=False)+"\n")

lens=[len(c["chunk"]) for c in dbt_chunks]
print(f"chunks: {len(dbt_chunks)} (header-first)")
print(f"median {int(statistics.median(lens))}, max {max(lens)}, broken-code {sum(c['chunk'].count(chr(96)*3)%2 for c in dbt_chunks)}")

# verify the fix: wizard answer should now be in ONE distinct chunk
wa=[i for i,c in enumerate(dbt_chunks) if "fct_monthly_revenue" in c["chunk"]]
print(f"wizard answer chunk(s): {wa} — {'FIXED (single clean chunk)' if len(wa)<=2 else 'still smeared'}")

chunks: 8169 (header-first)
median 792, max 19128, broken-code 23
wizard answer chunk(s): [2441] — FIXED (single clean chunk)


## 3. Index + embed (Day 3)

In [5]:
index = Index(text_fields=["chunk"], keyword_fields=["filename"])
index.fit(dbt_chunks)

embedding_model = SentenceTransformer(EMB_MODEL_NAME)
if os.path.exists(EMB_CACHE):
    embeddings = np.load(EMB_CACHE)
    print(f"loaded cached embeddings {embeddings.shape}")
else:
    texts=[c["chunk"] for c in dbt_chunks]
    embeddings=embedding_model.encode(texts, batch_size=32, show_progress_bar=True)
    np.save(EMB_CACHE, embeddings)
    print(f"encoded {embeddings.shape}")

assert embeddings.shape[0]==len(dbt_chunks), "count mismatch"
assert embeddings.shape[1]==embedding_model.encode("test").shape[0], "dim mismatch"

vindex = VectorSearch(keyword_fields=[])
vindex.fit(embeddings, dbt_chunks)
print("indexes ready")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/256 [00:00<?, ?it/s]

encoded (8169, 768)
indexes ready


## 4. Search + agent (Day 4)

In [6]:
def hybrid_search(query, num_results=5):
    lex=index.search(query, num_results=num_results)
    vec=vindex.search(embedding_model.encode(query), num_results=num_results)
    seen,merged=set(),[]
    for pair in zip(lex,vec):
        for r in pair:
            k=(r["filename"], r["chunk"][:50])
            if k not in seen: seen.add(k); merged.append(r)
    for r in lex+vec:
        k=(r["filename"], r["chunk"][:50])
        if k not in seen: seen.add(k); merged.append(r)
    return merged[:num_results]

def text_search(query, num_results=5):
    return [{"filename":r["filename"],"chunk":r["chunk"]} for r in hybrid_search(query,num_results)]

TOOLS=[{
    "name":"text_search",
    "description":("Search the dbt documentation. Call whenever you need factual info about dbt. "
                   "You may call it MULTIPLE times with different queries. Returns chunks with filenames."),
    "input_schema":{"type":"object","properties":{"query":{"type":"string"}},"required":["query"]},
}]
TOOL_FUNCTIONS={"text_search":text_search}
SYSTEM_PROMPT=("You are a dbt documentation assistant. Use text_search to ground every answer in the docs. "
               "Only state facts present in the search results — do not fill gaps with general knowledge. "
               "Cite the source filename for each claim. If the docs don't cover it, say so.")

def run_agent(question, max_turns=6, verbose=False):
    messages=[{"role":"user","content":question}]
    for turn in range(max_turns):
        resp=client.messages.create(model=MODEL, max_tokens=2048,
                                    system=SYSTEM_PROMPT, tools=TOOLS, messages=messages)
        if resp.stop_reason!="tool_use":
            return "".join(b.text for b in resp.content if b.type=="text")
        messages.append({"role":"assistant","content":resp.content})
        tr=[]
        for b in resp.content:
            if b.type=="tool_use":
                if verbose: print(f"  [turn {turn}] {b.input.get('query')!r}")
                tr.append({"type":"tool_result","tool_use_id":b.id,
                           "content":json.dumps(TOOL_FUNCTIONS[b.name](**b.input))})
        messages.append({"role":"user","content":tr})
    return "Stopped: max turns."

def ask_dbt(q): return run_agent(q, verbose=False)

print("agent ready:", callable(ask_dbt))

agent ready: True


## 5. Sanity check — the Wizard question that failed on Day 5

With header-first chunking, the answer chunk is now distinct and should surface.
Watch the trace and read whether the answer is about building a *dbt model* (right)
or configuring the *AI model* (the old wrong answer).

In [7]:
print(ask_dbt("Can I use Wizard to create a new model out of an existing SQL query, based on source tables?"))

Yes, you can use dbt Wizard to create a new model from existing SQL based on source tables! According to the dbt documentation on ["dbt Wizard use cases"](wizard-use-cases.md), here's how it works:

**What Wizard does:**

1. **Reads existing models/sources** — Wizard examines your project index to understand available columns in your existing staging models or source tables
2. **Generates SQL** — It creates the model SQL file with the logic you request
3. **Creates YAML configuration** — Wizard generates matching YAML blocks with tests and other metadata
4. **Shows you a diff** — You review and approve the changes before they're saved

**Example use case:**

You can ask Wizard to create a new model like this:

```text
Create a model called `fct_monthly_revenue` that joins `stg_orders` and `stg_payments`,
groups by `month` and `customer_id`, and materializes as a table. Add `not_null` tests
to the primary key and a unique test on the grain.
```

**Tips for success:**

- **Reference exis

## 6. Evaluate (Day 5) — measure whether the fix moved the number

In [8]:
def parse_json(text):
    text=text.strip()
    text=re.sub(r"^```(?:json)?","",text).strip(); text=re.sub(r"```$","",text).strip()
    m=re.search(r"[\[{].*[\]}]",text,re.S)
    if m: text=m.group(0)
    return json.loads(text)

def generate_questions(chunk, n=2):
    prompt=(f"Based ONLY on the documentation below, write {n} distinct questions a dbt user might ask "
            f"that THIS text answers. Natural and specific. Return ONLY a JSON list of strings.\n\n"
            f"DOCUMENTATION:\n{chunk['chunk'][:2000]}")
    resp=client.messages.create(model=MODEL, max_tokens=512, messages=[{"role":"user","content":prompt}])
    return [{"question":q,"source":chunk["filename"]} for q in parse_json(resp.content[0].text)]

def build_question_set(chunks, n_chunks=25, per_chunk=2, seed=42):
    if os.path.exists(QUESTIONS_FILE):
        print("loading cached questions"); return json.load(open(QUESTIONS_FILE,encoding="utf-8"))
    pool=[c for c in chunks if len(c["chunk"])>=500]; random.seed(seed)
    sampled=random.sample(pool, min(n_chunks,len(pool)))
    qs=[]
    for i,c in enumerate(sampled):
        try: qs.extend(generate_questions(c,per_chunk)); print(f"  {i+1}/{len(sampled)}")
        except Exception as e: print(f"  skip {i}: {e}")
        time.sleep(0.5)
    json.dump(qs, open(QUESTIONS_FILE,"w",encoding="utf-8"), indent=2)
    return qs

eval_questions=build_question_set(dbt_chunks)
eval_questions += [
    {"question":"How can I automate things in dbt?","source":"MULTI (broad)"},
    {"question":"Can I use Wizard to create a new model out of an existing SQL query, based on source tables?",
     "source":"website/docs/docs/dbt-ai/wizard-use-cases.md"},
]
print(f"questions: {len(eval_questions)}")

  1/25
  2/25
  3/25
  4/25
  5/25
  6/25
  7/25
  8/25
  9/25
  10/25
  11/25
  12/25
  13/25
  14/25
  15/25
  16/25
  17/25
  18/25
  19/25
  20/25
  21/25
  22/25
  23/25
  24/25
  25/25
questions: 52


In [9]:
def get_answers(questions):
    cache={}
    if os.path.exists(RESULTS_FILE):
        cache={r["question"]:r for r in json.load(open(RESULTS_FILE,encoding="utf-8"))}
    results=[]
    for i,item in enumerate(questions):
        q=item["question"]
        if q in cache and not cache[q]["answer"].startswith("ERROR"):
            results.append(cache[q]); continue
        try: answer=ask_dbt(q)
        except Exception as e: answer=f"ERROR: {e}"
        results.append({**item,"answer":answer}); print(f"  answered {i+1}/{len(questions)}")
        json.dump(results, open(RESULTS_FILE,"w",encoding="utf-8"), indent=2); time.sleep(0.5)
    return results

answered=get_answers(eval_questions)
print(f"answered {len(answered)}, errored {sum(r['answer'].startswith('ERROR') for r in answered)}")

  answered 1/52
  answered 2/52
  answered 3/52
  answered 4/52
  answered 5/52
  answered 6/52
  answered 7/52
  answered 8/52
  answered 9/52
  answered 10/52
  answered 11/52
  answered 12/52
  answered 13/52
  answered 14/52
  answered 15/52
  answered 16/52
  answered 17/52
  answered 18/52
  answered 19/52
  answered 20/52
  answered 21/52
  answered 22/52
  answered 23/52
  answered 24/52
  answered 25/52
  answered 26/52
  answered 27/52
  answered 28/52
  answered 29/52
  answered 30/52
  answered 31/52
  answered 32/52
  answered 33/52
  answered 34/52
  answered 35/52
  answered 36/52
  answered 37/52
  answered 38/52
  answered 39/52
  answered 40/52
  answered 41/52
  answered 42/52
  answered 43/52
  answered 44/52
  answered 45/52
  answered 46/52
  answered 47/52
  answered 48/52
  answered 49/52
  answered 50/52
  answered 51/52
  answered 52/52
answered 52, errored 0


In [10]:
JUDGE_PROMPT="""You are evaluating a dbt documentation assistant's answer.

QUESTION: {question}

ANSWER: {answer}

Score 1-5 (5=best): groundedness (factual, not made up), relevance (addresses the question),
citation (cites source filenames), completeness (fully answers).
Return ONLY: {{"groundedness":N,"relevance":N,"citation":N,"completeness":N,"comment":"one sentence"}}"""

def judge_answer(q,a):
    resp=client.messages.create(model=MODEL, max_tokens=300,
        messages=[{"role":"user","content":JUDGE_PROMPT.format(question=q,answer=a)}])
    return parse_json(resp.content[0].text)

def run_judge(answered):
    scored,fails=[],0
    for i,row in enumerate(answered):
        if row["answer"].startswith("ERROR"): continue
        try: scored.append({**row, **judge_answer(row["question"],row["answer"])})
        except Exception as e: fails+=1; print(f"  judge fail {i}: {e}")
        time.sleep(0.5)
    print(f"judged {len(scored)}, fails {fails}"); return scored

scored=run_judge(answered)

judged 52, fails 0


## 7. Results — before vs. after the chunking fix

In [11]:
df=pd.DataFrame(scored)
criteria=["groundedness","relevance","citation","completeness"]
print("=== mean scores (header-first chunking) ===")
print(df[criteria].mean().round(2).to_string())
print(f"\noverall mean: {df[criteria].mean().mean():.2f}")
print("\n(Day 5 sliding-window baseline was 3.64 — compare.)")

# the Wizard question specifically
wiz=df[df["question"].str.contains("Wizard")]
if len(wiz):
    print("\n=== Wizard question (was groundedness 2 on Day 5) ===")
    print(wiz[["groundedness","relevance","citation","completeness"]].to_string(index=False))

=== mean scores (header-first chunking) ===
groundedness    3.46
relevance       4.42
citation        3.54
completeness    3.81

overall mean: 3.81

(Day 5 sliding-window baseline was 3.64 — compare.)

=== Wizard question (was groundedness 2 on Day 5) ===
 groundedness  relevance  citation  completeness
            2          5         2             4
            5          5         5             5
            2          4         3             2
            2          4         3             3
            3          4         2             3


## Findings

### The number moved — measurably
| Metric | Sliding-window (Day 5) | Header-first (pipeline) |
|--------|------------------------|-------------------------|
| groundedness | 3.37 | 3.46 |
| relevance | 4.04 | 4.42 |
| citation | 3.67 | 3.54 |
| completeness | 3.48 | 3.81 |
| **overall** | **3.64** | **3.81** |

Header-first chunking raised the overall score and notably improved relevance
(4.04 → 4.42) and completeness (3.48 → 3.81).

### The loop closed — eval finding traced to a root cause three days upstream
The Wizard question ("create a new model from an existing SQL query") scored
groundedness **2** on Day 5. Tracing it:
1. The agent's searches were well-formed (4 targeted queries, correct terminology).
2. The answer *was* in the corpus (`wizard-use-cases.md`, "Build a new model").
3. But Day 2's sliding-window chunking had split that file into 18 overlapping,
   near-duplicate fragments — 11 sharing identical opening lines. The answer chunk
   was indistinguishable from its look-alikes, so search couldn't surface it.
4. The agent synthesized from the chunks that *did* rank (a keyword-dense overview
   page about configuring Wizard's AI model) and answered the wrong sense of "model."

Header-first chunking (split on `##`/`###`, one distinct section per chunk) isolated
the answer into a single clean chunk. **This is why evaluation exists: it points at
root causes upstream, not just symptoms.**

### But "fixed" is too strong — the honest result
Across five runs, the Wizard question scored groundedness **2 to 5**. Header-first
chunking made the correct answer *achievable* (one run scored 5/5/5/5, which the
sliding-window version never produced) but not *reliable*. The chunking fix removed
the structural blocker; the remaining variance is a retrieval-**ranking** problem —
the right chunk exists and is retrievable, but doesn't consistently rank top-5
depending on the agent's self-chosen query wording.

### What I learned about the system
1. **Retrieval quality bottlenecks answer quality, not the LLM.** The Wizard answer
   flipped from wrong to right purely by changing what was retrieved — same model,
   same prompt.
2. **Vocabulary mismatch is real.** Lexical search failed this question at any depth
   because the query ("create a model from SQL") and answer ("build a mart joining
   staging tables") shared almost no words. Vector search found it at rank 1.
3. **Agentic search helps breadth, can hurt precision.** The broad "automate dbt"
   question — hopeless for single-shot RAG — scored well because the agent ran
   multiple searches and synthesized. But on the lexically ambiguous Wizard question,
   the agent's freedom to phrase its own query sometimes retrieved the wrong sense.
4. **Chunking decisions echo downstream.** A Day 2 choice determined a Day 5 score.

### Known next improvements (not built here)
- **Reranking** to consistently surface the right chunk — the fix for the remaining
  Wizard variance.
- **Citation prompt** — citation (3.54) is now the weakest criterion; the agent
  answers correctly but inconsistently names sources. A system-prompt fix.
- **Practical deployment** would point this pipeline at *private* data (internal
  docs, a real dbt project) — where a general model can't reach — rather than public
  docs it already knows.

### What this project actually demonstrates
Not "I built a RAG agent" but: **I built one, measured it, traced its weakest score
to a root cause three days upstream, fixed the cause, re-measured, and honestly
assessed what remained.** The pipeline — ingest → chunk → search → agent → eval — is
the transferable asset; the dbt corpus was the practice ground.